# WheelPose — Prediction Visualization

Use this notebook to:
1. Visually inspect keypoint predictions on wheelchair basketball images
2. Compare ImageNet baseline vs fine-tuned WheelPose-Opt
3. Spot failure cases (hips, occluded legs) to document in report

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

from configs.register_datasets import register_all
from configs.model_config import get_config

register_all()

In [ ]:
# ── Load both predictors ──────────────────────────────────────────────────

# Baseline: ImageNet pretrained (no fine-tuning)
cfg_baseline = get_config()
cfg_baseline.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor_baseline = DefaultPredictor(cfg_baseline)

# Fine-tuned: WheelPose-Opt
# [PLACEHOLDER] Update path once training is complete
FINETUNED_WEIGHTS = "../outputs/wheelpose_opt/model_final.pth"
cfg_ft = get_config(weights=FINETUNED_WEIGHTS)
cfg_ft.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor_ft = DefaultPredictor(cfg_ft)

metadata = MetadataCatalog.get('wheelpose_test')
print('Both predictors loaded.')

In [ ]:
def show_comparison(img_path):
    """Show ImageNet vs Fine-tuned predictions side by side."""
    img = cv2.imread(img_path)
    assert img is not None, f"Cannot read {img_path}"

    out_base = predictor_baseline(img)
    out_ft   = predictor_ft(img)

    vis_base = Visualizer(img[:,:,::-1], metadata=metadata, scale=1.0)
    vis_ft   = Visualizer(img[:,:,::-1], metadata=metadata, scale=1.0)

    drawn_base = vis_base.draw_instance_predictions(out_base['instances'].to('cpu')).get_image()
    drawn_ft   = vis_ft.draw_instance_predictions(out_ft['instances'].to('cpu')).get_image()

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(drawn_base);  axes[0].set_title('ImageNet Baseline',   fontsize=14)
    axes[1].imshow(drawn_ft);    axes[1].set_title('WheelPose Fine-tuned', fontsize=14)
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Browse test images ────────────────────────────────────────────────────
from pathlib import Path
import random

test_images = sorted(Path('../data/real_world/images').glob('*.jpg'))
print(f'Found {len(test_images)} test images')

# Show a random sample
sample = random.choice(test_images)
show_comparison(str(sample))

In [ ]:
# ── Per-keypoint confidence inspection ───────────────────────────────────
# Run fine-tuned model on all test images and plot avg confidence per keypoint
from configs.register_datasets import KEYPOINT_NAMES
import json

# [PLACEHOLDER] Point to your extracted keypoints JSON after running extract_keypoints.py
# json_path = '../outputs/keypoints_output.json'
# with open(json_path) as f:
#     kp_data = json.load(f)
#
# avg_conf = {name: [] for name in KEYPOINT_NAMES}
# for frame in kp_data:
#     for person in frame['persons']:
#         for name, vals in person['keypoints'].items():
#             avg_conf[name].append(vals['confidence'])
#
# means = {k: np.mean(v) for k, v in avg_conf.items() if v}
# plt.figure(figsize=(12, 4))
# plt.bar(means.keys(), means.values())
# plt.xticks(rotation=45, ha='right')
# plt.ylabel('Avg Confidence')
# plt.title('Per-Keypoint Confidence — Fine-tuned WheelPose')
# plt.tight_layout()
# plt.show()

print('Uncomment the block above once keypoints JSON is available.')